# RF-DETR Multi-Category Detection — Full Pipeline v6

**Purpose:** Build a multi-category dataset (supporting `location_tag` and other categories) from JSON records containing Azure Blob image URLs, create stratified train/val/test COCO datasets, train RF-DETR Base, select the best checkpoint, run test inference, visualize predictions, and save results.

### Pipeline features & configuration
- Uses the **installed RF-DETR `TrainConfig` API**: `dataset_dir` + `dataset_file`.
- Configured with working training hyperparameters: `batch_size=2`, `lr=1e-4`, `num_workers=2`, `val_split="val"`.
- Keeps `images/` inside each split directory.
- Uses **one tqdm bar per major stage**; no nested per-image/per-bbox progress bars.
- Azure image downloads use a thread pool because the operation is I/O-bound.
- Image dimensions are read once and cached.
- COCO annotation generation reuses cached dimensions and does not reopen every image.
- **Multi-category support**: automatically detects or filters categories (`location_tag`, `blue_aisle`, etc.) and maps them to 0-indexed category IDs (`0, 1, ...`).
- **Multi-label category stratification**: guarantees equal proportions of every `category_id` across train, val, and test splits.


In [ ]:

# STEP 0 — Install Required Versions (RF-DETR 1.4.0 & CUDA 12.1 Stack)
# 1. PyTorch Stack with CUDA 12.1 (cu121)
!pip install torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

# 2. Strict Core Packages (rfdetr==1.4.0, supervision>=0.22.0, pycocotools>=2.0.7)
#    Other packages allow flexible patch/minor versions (e.g. 0.1.x / X.Y.*) to avoid resolver conflicts.
!pip install "rfdetr==1.4.0" "supervision>=0.22.0" "pycocotools>=2.0.7" "pytorch-lightning>=2.5.0" "numpy>=1.24.0,<2.0.0" "pandas>=2.0.0" "scikit-learn>=1.3.0" "opencv-python>=4.8.0" "albumentations>=1.3.0" "transformers>=4.40.0" "timm>=0.9.0" "accelerate>=0.28.0" "roboflow>=1.3.0" "rf100vl>=1.1.0"


In [ ]:

# CELL 1 — Imports
import os
import re
import json
import math
import random
import shutil
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from dotenv import load_dotenv
import supervision as sv

from azure.storage.blob import BlobServiceClient

import torch
from rfdetr import RFDETRBase

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Supervision:", sv.__version__)


In [ ]:

# CELL 2 — Configuration
INPUT_JSON_DIR = Path("./coco_files")
DATASET_DIR = Path("./rfdetr_dataset")
OUTPUT_DIR = Path("./rfdetr_output")
INFERENCE_OUTPUT_DIR = Path("./inference_outputs")

PRETRAINED_WEIGHTS = "/home/jupyter/rf-detr-base-coco.pth"
AZURE_CONNECTION_STRING_ENV = "AZURE_STORAGE_CONNECTION_STRING"

IMAGE_FIELD = "image_id"
CATEGORY_FIELD = "category_id"
BBOX_FIELD = "bbox"
BBOX_FORMAT = "xywh"

# Categories configuration:
# - Set TARGET_CLASSES = None to automatically include ALL categories found in the JSON annotations
# - Or provide an explicit list to filter, e.g.: TARGET_CLASSES = ["location_tag", "other_category"]
TARGET_CLASSES = None

# Resolution & Device (updated to resolution 560)
RESOLUTION = 560
DEVICE = "cuda"

# Optimizer & Training Hyperparameters
OPTIMIZER = "adam"
LR_SCHEDULER = "cosine"
EPOCHS = 50
BATCH_SIZE = 2        # Safer for T4 GPUs (prevents CUDA OOM)
LR = 1e-4             # Learning rate
NUM_WORKERS = 2       # Dataloader workers
GRAD_ACCUM_STEPS = 1

TRAIN_RATIO = 0.80
VALID_RATIO = 0.10
TEST_RATIO = 0.10
RANDOM_SEED = 42

DOWNLOAD_WORKERS = 16
# Default Confidence Threshold and NMS Post-Processing Threshold
CONFIDENCE = 0.50
NMS_THRESHOLD = 0.50

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

assert abs(TRAIN_RATIO + VALID_RATIO + TEST_RATIO - 1.0) < 1e-8
assert BBOX_FORMAT == "xywh"

print("Configuration loaded (Resolution 560, Adam, Cosine Decay, Conf 0.50, NMS 0.50).")


In [ ]:

# CELL 3 — Create directories and load Azure credentials
for p in [DATASET_DIR, OUTPUT_DIR, INFERENCE_OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

load_dotenv()
connection_string = os.getenv(AZURE_CONNECTION_STRING_ENV)

if not connection_string:
    raise RuntimeError(
        f"Azure connection string not found. Set {AZURE_CONNECTION_STRING_ENV} in .env."
    )

blob_service_client = BlobServiceClient.from_connection_string(connection_string)

print("Azure client initialized.")


In [ ]:

# CELL 4 — Read all annotation JSON files (supports standard JSON & JSONLines)
json_files = sorted(INPUT_JSON_DIR.glob("*.json"))

if not json_files:
    raise FileNotFoundError(f"No JSON files found in {INPUT_JSON_DIR.resolve()}")

records = []

with tqdm(total=len(json_files), desc="Reading annotation JSON files", unit="file") as pbar:
    for json_file in json_files:
        with json_file.open("r", encoding="utf-8") as f:
            content = f.read().strip()

        # 1. Try parsing as a single JSON structure (list or dict)
        parsed = False
        try:
            data = json.loads(content)
            if isinstance(data, list):
                for item in data:
                    if isinstance(item, list):
                        records.extend(item)
                    elif isinstance(item, dict):
                        records.append(item)
            elif isinstance(data, dict):
                if "annotations" in data and isinstance(data["annotations"], list):
                    records.extend(data["annotations"])
                else:
                    records.append(data)
            parsed = True
        except Exception:
            parsed = False

        # 2. Fallback: Parse line-by-line (NDJSON / JSONLines / multiple JSON lists)
        if not parsed:
            for line in content.splitlines():
                line = line.strip()
                if not line:
                    continue
                try:
                    item = json.loads(line)
                    if isinstance(item, list):
                        records.extend(item)
                    elif isinstance(item, dict):
                        records.append(item)
                except Exception:
                    pass
        pbar.update(1)

print(f"JSON files processed: {len(json_files)}")
print(f"Raw records extracted: {len(records)}")


In [ ]:

# CELL 5 — Normalize annotations and categorize (supports list categories e.g. ['location_tag'], ['blue_aisle'])
def normalize_category(value):
    if isinstance(value, (list, tuple)):
        if len(value) == 0:
            return None
        value = value[0]
    if value is None:
        return None
    cleaned = str(value).strip()
    return cleaned if cleaned else None

normalized = []
skipped = 0

with tqdm(total=len(records), desc="Processing annotations", unit="record") as pbar:
    for rec in records:
        try:
            image_url = str(rec.get(IMAGE_FIELD, "")).strip()
            category = normalize_category(rec.get(CATEGORY_FIELD))
            bbox = rec.get(BBOX_FIELD)

            if not category or not image_url:
                skipped += 1
                pbar.update(1)
                continue

            # Filter if an explicit list was specified in TARGET_CLASSES
            if TARGET_CLASSES is not None and category not in TARGET_CLASSES:
                skipped += 1
                pbar.update(1)
                continue

            if not isinstance(bbox, (list, tuple)) or len(bbox) != 4:
                skipped += 1
                pbar.update(1)
                continue

            x, y, w, h = map(float, bbox)

            if w <= 0 or h <= 0:
                skipped += 1
                pbar.update(1)
                continue

            normalized.append({
                "image_url": image_url,
                "category": category,
                "bbox": [x, y, w, h],
            })
        except Exception:
            skipped += 1
        pbar.update(1)

# Build category mappings (0-indexed for RF-DETR)
if TARGET_CLASSES is not None:
    CATEGORIES = list(TARGET_CLASSES)
else:
    CATEGORIES = sorted(list({item["category"] for item in normalized}))

CATEGORY_TO_ID = {cat: idx for idx, cat in enumerate(CATEGORIES)}
ID_TO_CATEGORY = {idx: cat for cat, idx in CATEGORY_TO_ID.items()}
NUM_CLASSES = len(CATEGORIES)

category_counts = defaultdict(int)
for item in normalized:
    category_counts[item["category"]] += 1

print(f"Kept records: {len(normalized)}")
print(f"Skipped records: {skipped}")
print(f"Total unique classes: {NUM_CLASSES}")
print(f"Categories: {CATEGORIES}")
print(f"Category ID mapping: {CATEGORY_TO_ID}")
print(f"Class distribution: {dict(category_counts)}")

assert NUM_CLASSES >= 1, "At least one category must be present in the dataset."


In [ ]:

# CELL 6 — Group annotations by image URL
image_records = defaultdict(list)

for item in normalized:
    image_records[item["image_url"]].append({
        "bbox": item["bbox"],
        "category": item["category"],
        "category_id": CATEGORY_TO_ID[item["category"]],
    })

image_urls = sorted(image_records.keys())

print(f"Unique images: {len(image_urls)}")
print(f"Images with annotations: {sum(bool(v) for v in image_records.values())}")


In [ ]:

# CELL 7 — Azure URL helpers
def parse_blob_url(url):
    clean_url = url.split("?", 1)[0]
    match = re.match(r"https?://([^/]+)/(.+)", clean_url)

    if not match:
        raise ValueError(f"Invalid Azure Blob URL: {url}")

    account_host = match.group(1)
    blob_path = match.group(2)

    parts = blob_path.split("/", 1)
    if len(parts) != 2:
        raise ValueError(f"Invalid blob path: {blob_path}")

    container_name, blob_name = parts
    return account_host, container_name, blob_name

def safe_filename_from_url(url, index):
    clean = url.split("?", 1)[0]
    name = Path(clean).name
    if not name or Path(name).suffix.lower() not in IMAGE_EXTENSIONS:
        name = f"image_{index:08d}.jpg"

    # Keep filenames filesystem-safe.
    name = re.sub(r"[^A-Za-z0-9._-]", "_", name)
    return f"{index:08d}_{name}"


In [ ]:

# CELL 8 — Download one image
def download_one(args):
    index, url = args

    try:
        _, container_name, blob_name = parse_blob_url(url)
        blob_client = blob_service_client.get_blob_client(
            container=container_name,
            blob=blob_name
        )

        filename = safe_filename_from_url(url, index)
        output_path = DATASET_DIR / "downloaded_images" / filename
        output_path.parent.mkdir(parents=True, exist_ok=True)

        if not output_path.exists():
            data = blob_client.download_blob().readall()
            output_path.write_bytes(data)

        with Image.open(output_path) as img:
            width, height = img.size

        return {
            "url": url,
            "path": str(output_path),
            "filename": filename,
            "width": width,
            "height": height,
            "error": None,
        }

    except Exception as e:
        return {
            "url": url,
            "path": None,
            "filename": None,
            "width": None,
            "height": None,
            "error": str(e),
        }


In [ ]:

# CELL 9 — Parallel Azure download
download_results = {}

with tqdm(total=len(image_urls), desc="Downloading images from Azure", unit="stage") as pbar:
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
        futures = {
            executor.submit(download_one, (i, url)): url
            for i, url in enumerate(image_urls)
        }

        for future in as_completed(futures):
            result = future.result()
            download_results[result["url"]] = result
            pbar.update(1)

download_errors = {
    url: r["error"]
    for url, r in download_results.items()
    if r["error"]
}

print(f"Downloaded/available: {len(download_results) - len(download_errors)}")
print(f"Download errors: {len(download_errors)}")

if download_errors:
    print("First download error:", next(iter(download_errors.values())))


In [ ]:

# CELL 10 — Build train/val/test split with Category Proportions Maintained
# 1. First calculate baseline proportions of each category across the whole dataset.
# 2. Maintain those exact category proportions within each split (train, val, test).

def multilabel_stratified_split(image_to_categories, ratios, seed=42):
    """
    Performs iterative stratification for multi-label / multi-class object detection.
    Guarantees balanced category proportions across splits according to ratios.
    """
    rng = random.Random(seed)
    split_names = list(ratios.keys())
    all_images = sorted(image_to_categories.keys())
    rng.shuffle(all_images)

    all_categories = sorted(list({c for cats in image_to_categories.values() for c in cats}))

    # Count total instances per category and per image
    cat_total = defaultdict(int)
    img_cat_counts = {}
    for img in all_images:
        counts = defaultdict(int)
        for c in image_to_categories[img]:
            counts[c] += 1
            cat_total[c] += 1
        img_cat_counts[img] = counts

    total_annotations = sum(cat_total.values())
    cat_global_proportions = {c: cat_total[c] / max(1, total_annotations) for c in all_categories}

    # Target counts per split per category (Ratio_s * Count_c)
    targets = {s: {c: cat_total[c] * ratios[s] for c in all_categories} for s in split_names}
    current = {s: {c: 0 for c in all_categories} for s in split_names}
    assigned = {s: [] for s in split_names}
    unassigned = set(all_images)

    # Allocate rarest categories first to prevent starving minority classes in val/test
    sorted_cats = sorted(all_categories, key=lambda c: cat_total[c])

    for c in sorted_cats:
        imgs_with_c = [img for img in all_images if img in unassigned and img_cat_counts[img][c] > 0]
        imgs_with_c.sort(key=lambda img: sum(img_cat_counts[img].values()))

        for img in imgs_with_c:
            if img not in unassigned:
                continue
            # Select the split with the greatest relative deficit for this category
            best_split = max(
                split_names,
                key=lambda s: (targets[s][c] - current[s][c]) / max(1e-6, targets[s][c])
            )
            assigned[best_split].append(img)
            unassigned.remove(img)
            for cat, cnt in img_cat_counts[img].items():
                current[best_split][cat] += cnt

    # Assign any remaining images (e.g. background images with no annotations)
    for img in sorted(unassigned):
        best_split = min(split_names, key=lambda s: len(assigned[s]) / ratios[s])
        assigned[best_split].append(img)

    return assigned, current, cat_total, cat_global_proportions

# Collect downloaded and valid image URLs
valid_urls = [
    url for url in image_urls
    if url in download_results and download_results[url]["error"] is None
]

# Map each image URL to its list of categories
image_to_categories = {
    url: [item["category"] for item in image_records[url]]
    for url in valid_urls
}

# Perform stratified split
SPLITS, split_cat_counts, cat_totals, cat_proportions = multilabel_stratified_split(
    image_to_categories=image_to_categories,
    ratios={"train": TRAIN_RATIO, "val": VALID_RATIO, "test": TEST_RATIO},
    seed=RANDOM_SEED
)

total_annotations = sum(cat_totals.values())
split_ann_totals = {s: sum(split_cat_counts[s].values()) for s in ["train", "val", "test"]}

# TABLE 1: Category Proportions Verification Table (Category Mix % inside each split)
print("=" * 85)
print("CATEGORY PROPORTIONS VERIFICATION TABLE: Category Mixture in Each Split")
print("=" * 85)
h1 = f"{'Category':<22} {'Total Count':<12} {'Dataset Mix%':<14} {'Train Mix%':<14} {'Val Mix%':<14} {'Test Mix%':<14}"
print(h1)
print("-" * 85)
for cat in CATEGORIES:
    tot = cat_totals[cat]
    d_pct = (cat_proportions[cat] * 100) if total_annotations > 0 else 0
    tr_pct = (split_cat_counts['train'][cat] / split_ann_totals['train'] * 100) if split_ann_totals['train'] > 0 else 0
    va_pct = (split_cat_counts['val'][cat] / split_ann_totals['val'] * 100) if split_ann_totals['val'] > 0 else 0
    te_pct = (split_cat_counts['test'][cat] / split_ann_totals['test'] * 100) if split_ann_totals['test'] > 0 else 0
    print(f"{cat:<22} {tot:<12} {d_pct:>6.2f}%        {tr_pct:>6.2f}%        {va_pct:>6.2f}%        {te_pct:>6.2f}%")
print("-" * 85)
print(f"{'Total Annotations':<22} {total_annotations:<12} {'100.00%':<14} {'100.00%':<14} {'100.00%':<14} {'100.00%':<14}")
print("=" * 85 + "\n")

# TABLE 2: Split Allocation Counts (Count and % of total instances assigned to each split)
print("=" * 80)
print("SPLIT ALLOCATION COUNTS: Distribution Across Train / Val / Test")
print("=" * 80)
h2 = f"{'Category':<22} {'Total':<8} {'Train (' + str(int(TRAIN_RATIO*100)) + '%)':<16} {'Val (' + str(int(VALID_RATIO*100)) + '%)':<16} {'Test (' + str(int(TEST_RATIO*100)) + '%)':<16}"
print(h2)
print("-" * 80)
for cat in CATEGORIES:
    tot = cat_totals[cat]
    row = f"{cat:<22} {tot:<8} "
    for s in ["train", "val", "test"]:
        cnt = split_cat_counts[s][cat]
        pct = (cnt / tot * 100) if tot > 0 else 0
        row += f"{cnt} ({pct:.1f}%)".ljust(16) + " "
    print(row)
print("-" * 80)
img_row = f"{'Total Images':<22} {len(valid_urls):<8} "
for s in ["train", "val", "test"]:
    cnt = len(SPLITS[s])
    pct = (cnt / len(valid_urls) * 100) if len(valid_urls) > 0 else 0
    img_row += f"{cnt} ({pct:.1f}%)".ljust(16) + " "
print(img_row)
print("=" * 80)


In [ ]:

# CELL 11 — Create RF-DETR dataset folders
for split in SPLITS:
    (DATASET_DIR / split / "images").mkdir(parents=True, exist_ok=True)

# Create 'valid' alias to 'val' for backward compatibility
val_dir = DATASET_DIR / "val"
valid_dir = DATASET_DIR / "valid"
if val_dir.exists() and not valid_dir.exists():
    try:
        valid_dir.symlink_to("val", target_is_directory=True)
    except Exception:
        pass

with tqdm(total=1, desc="Preparing dataset folders", unit="stage") as pbar:
    pbar.update(1)

print("Dataset folders ready (with val/ and valid/ compatibility).")


In [ ]:

# CELL 12 — Copy images into split/images and cache dimensions
IMAGE_METADATA = {}

with tqdm(total=1, desc="Preparing split images", unit="stage") as pbar:
    for split, urls in SPLITS.items():
        for url in urls:
            result = download_results[url]
            src = Path(result["path"])
            dst = DATASET_DIR / split / "images" / result["filename"]

            if src.resolve() != dst.resolve():
                if not dst.exists():
                    shutil.copy2(src, dst)

            IMAGE_METADATA[url] = {
                "filename": result["filename"],
                "width": int(result["width"]),
                "height": int(result["height"]),
                "path": str(dst),
            }
    pbar.update(1)

print(f"Cached image metadata: {len(IMAGE_METADATA)}")


In [ ]:

# CELL 13 — COCO dataset builder
def clip_xywh(x, y, w, h, width, height):
    x1 = max(0.0, min(float(x), float(width)))
    y1 = max(0.0, min(float(y), float(height)))
    x2 = max(0.0, min(float(x + w), float(width)))
    y2 = max(0.0, min(float(y + h), float(height)))

    clipped_w = x2 - x1
    clipped_h = y2 - y1

    return x1, y1, clipped_w, clipped_h

def build_coco(split, urls):
    images = []
    annotations = []

    url_to_image_id = {}

    for image_id, url in enumerate(urls, start=1):
        meta = IMAGE_METADATA[url]

        # RF-DETR receives the split directory as the dataset root,
        # therefore file_name includes the images/ subdirectory.
        file_name = f"images/{meta['filename']}"

        images.append({
            "id": image_id,
            "file_name": file_name,
            "width": meta["width"],
            "height": meta["height"],
        })
        url_to_image_id[url] = image_id

    ann_id = 1

    for url in urls:
        meta = IMAGE_METADATA[url]
        image_id = url_to_image_id[url]

        for item in image_records[url]:
            x, y, w, h = item["bbox"]
            cat_id = item["category_id"]
            x, y, w, h = clip_xywh(
                x, y, w, h,
                meta["width"],
                meta["height"]
            )

            if w <= 0 or h <= 0:
                continue

            annotations.append({
                "id": ann_id,
                "image_id": image_id,
                "category_id": cat_id,
                "bbox": [x, y, w, h],
                "area": w * h,
                "iscrowd": 0,
            })
            ann_id += 1

    coco_categories = [
        {
            "id": cat_id,
            "name": cat_name,
            "supercategory": "object",
        }
        for cat_name, cat_id in sorted(CATEGORY_TO_ID.items(), key=lambda x: x[1])
    ]

    return {
        "info": {
            "description": "RF-DETR multi-category dataset"
        },
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": coco_categories,
    }


In [ ]:

# CELL 14 — Generate COCO JSON files & create backup
for split, urls in SPLITS.items():
    coco = build_coco(split, urls)
    output_json = DATASET_DIR / split / "_annotations.coco.json"

    with output_json.open("w", encoding="utf-8") as f:
        json.dump(coco, f, separators=(",", ":"))

# Create a backup copy of all generated COCO annotations
backup_dir = DATASET_DIR.parent / f"{DATASET_DIR.name}_backup_annotations"
backup_dir.mkdir(parents=True, exist_ok=True)
for split in SPLITS:
    src_file = DATASET_DIR / split / "_annotations.coco.json"
    if src_file.exists():
        dst_file = backup_dir / f"{split}_annotations.coco.json"
        shutil.copy2(src_file, dst_file)

with tqdm(total=1, desc="Writing COCO annotations & backup", unit="stage") as pbar:
    pbar.update(1)

print("COCO files generated.")
print(f"Annotation backups saved to: {backup_dir.resolve()}")


In [ ]:

# CELL 15 — Dataset validation
dataset_summary = {}

with tqdm(total=1, desc="Validating COCO dataset", unit="stage") as pbar:
    for split in SPLITS:
        split_dir = DATASET_DIR / split
        annotation_file = split_dir / "_annotations.coco.json"

        with annotation_file.open("r", encoding="utf-8") as f:
            coco = json.load(f)

        image_count = len(coco["images"])
        annotation_count = len(coco["annotations"])

        missing_images = [
            img["file_name"]
            for img in coco["images"]
            if not (split_dir / img["file_name"]).exists()
        ]

        class_distribution = defaultdict(int)
        for ann in coco["annotations"]:
            cat_id = ann["category_id"]
            cat_name = ID_TO_CATEGORY.get(cat_id, f"class_{cat_id}")
            class_distribution[cat_name] += 1

        dataset_summary[split] = {
            "images": image_count,
            "annotations": annotation_count,
            "class_distribution": dict(class_distribution),
            "missing_images": len(missing_images),
        }

        if missing_images:
            raise FileNotFoundError(
                f"{split}: {len(missing_images)} image files are missing."
            )
    pbar.update(1)

print(json.dumps(dataset_summary, indent=2))


In [ ]:

# CELL 16 — Check pretrained weights
weights_path = Path(PRETRAINED_WEIGHTS)

if not weights_path.exists():
    raise FileNotFoundError(
        f"Pretrained RF-DETR weights not found: {weights_path}"
    )

print(f"Pretrained weights: {weights_path}")
print(f"Exists: {weights_path.exists()}")


In [ ]:

# CELL 17 — Initialize RF-DETR Base
model = RFDETRBase(
    pretrain_weights=PRETRAINED_WEIGHTS,
    resolution=RESOLUTION,
    device=DEVICE,
    num_classes=NUM_CLASSES
)

print("RF-DETR model initialized.")
print("Resolution:", RESOLUTION)
print("Num classes:", NUM_CLASSES)
print("Categories:", CATEGORIES)
print("Device:", DEVICE)


In [ ]:

# CELL 18 — Training configuration (Adam, Cosine Decay, Resolution 560)
train_kwargs = {
    "dataset_dir": str(DATASET_DIR),
    "train_split": "train",
    "val_split": "val",
    "annotation_file": "_annotations.coco.json",
    "image_folder": "images",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "optimizer": OPTIMIZER,
    "lr_scheduler": LR_SCHEDULER,
    "num_workers": NUM_WORKERS,
    "output_dir": str(OUTPUT_DIR),
}

print(json.dumps(train_kwargs, indent=2))


In [ ]:

# CELL 19 — Train (with Adam & Cosine Scheduler)
try:
    model.train(**train_kwargs)
except TypeError as e:
    print(f"Direct kwargs warning: {e}. Retrying with core training kwargs...")
    core_kwargs = {
        "dataset_dir": str(DATASET_DIR),
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "optimizer": OPTIMIZER,
        "lr_scheduler": LR_SCHEDULER,
        "num_workers": NUM_WORKERS,
        "output_dir": str(OUTPUT_DIR),
    }
    model.train(**core_kwargs)


In [ ]:

# CELL 20 — Find checkpoints
checkpoint_candidates = []

with tqdm(total=1, desc="Searching training checkpoints", unit="stage") as pbar:
    for pattern in ["*.pth", "*.pt", "*.ckpt"]:
        checkpoint_candidates.extend(OUTPUT_DIR.rglob(pattern))
    pbar.update(1)

checkpoint_candidates = sorted(
    set(checkpoint_candidates),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

print(f"Checkpoint files found: {len(checkpoint_candidates)}")

for p in checkpoint_candidates[:20]:
    print(p)


In [ ]:

# CELL 21 — Select best checkpoint
preferred_names = [
    "checkpoint_best_regular.pth",
    "checkpoint_best_total.pth",
    "checkpoint_best.pth",
    "best.pth",
    "checkpoint_best.pt",
    "best.pt",
]

BEST_CHECKPOINT = None

for name in preferred_names:
    matches = list(OUTPUT_DIR.rglob(name))
    if matches:
        BEST_CHECKPOINT = matches[0]
        break

if BEST_CHECKPOINT is None and checkpoint_candidates:
    BEST_CHECKPOINT = checkpoint_candidates[0]

if BEST_CHECKPOINT is None:
    raise FileNotFoundError(
        "No trained checkpoint was found in the RF-DETR output directory."
    )

print("Selected checkpoint:", BEST_CHECKPOINT)


In [ ]:

# CELL 22 — Load trained RF-DETR model
trained_model = RFDETRBase(
    pretrain_weights=str(BEST_CHECKPOINT),
    resolution=RESOLUTION,
    device=DEVICE,
    num_classes=NUM_CLASSES
)

# Optimize inference runtime if supported
try:
    trained_model.optimize_for_inference()
    print("Model optimized for inference.")
except Exception as e:
    print(f"Notice: optimize_for_inference() skipped ({e})")

print("Trained model loaded.")
print("Num classes:", NUM_CLASSES)
print("Categories:", CATEGORIES)


In [ ]:

# CELL 23 — Test inference
# Inference is run without a per-image tqdm bar to keep notebook output clean.

test_images = []

with tqdm(total=1, desc="Collecting test images", unit="stage") as pbar:
    for url in SPLITS["test"]:
        meta = IMAGE_METADATA[url]
        test_images.append(Path(meta["path"]))
    pbar.update(1)

print(f"Test images: {len(test_images)}")


In [ ]:

# CELL 24 — Run inference, apply NMS post-processing, annotate with Supervision, and save predictions
annotated_dir = INFERENCE_OUTPUT_DIR / "annotated_images"
annotated_dir.mkdir(parents=True, exist_ok=True)

# Color palette for multi-class visualization
color_palette = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9bee", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

prediction_records = []

with tqdm(total=len(test_images), desc="Running test inference & NMS post-processing", unit="image") as pbar:
    for image_path in test_images:
        image_pil = Image.open(image_path).convert("RGB")

        # Predict with RF-DETR using default confidence threshold (0.50)
        detections = trained_model.predict(
            str(image_path),
            threshold=CONFIDENCE
        )

        # Post-Processing: Non-Maximum Suppression (NMS) to refine duplicate detections
        try:
            if hasattr(detections, "with_nms"):
                detections = detections.with_nms(threshold=NMS_THRESHOLD)
            elif hasattr(sv, "non_max_suppression"):
                detections = sv.non_max_suppression(detections, threshold=NMS_THRESHOLD)
        except Exception:
            pass

        # Compute optimal text scale and line thickness based on image dimensions
        text_scale = sv.calculate_optimal_text_scale(resolution_wh=image_pil.size)
        thickness = sv.calculate_optimal_line_thickness(resolution_wh=image_pil.size)

        box_annotator = sv.BoxAnnotator(
            color=color_palette,
            thickness=thickness
        )
        label_annotator = sv.LabelAnnotator(
            color=color_palette,
            text_color=sv.Color.BLACK,
            text_scale=text_scale,
            smart_position=True
        )

        # Build descriptive labels: '<class_name> <confidence>'
        labels = []
        if hasattr(detections, "class_id") and detections.class_id is not None:
            labels = [
                f"{ID_TO_CATEGORY.get(int(cid), f'class_{cid}')} {conf:.2f}"
                for cid, conf in zip(detections.class_id, detections.confidence)
            ]

        # Annotate image with bounding boxes and labels
        annotated_frame = np.array(image_pil)
        if hasattr(detections, "xyxy") and len(detections.xyxy) > 0:
            annotated_frame = box_annotator.annotate(
                scene=annotated_frame,
                detections=detections
            )
            if labels:
                annotated_frame = label_annotator.annotate(
                    scene=annotated_frame,
                    detections=detections,
                    labels=labels
                )

        # Save annotated image to disk
        annotated_image = Image.fromarray(annotated_frame)
        annotated_save_path = annotated_dir / image_path.name
        annotated_image.save(annotated_save_path)

        prediction_records.append({
            "image": str(image_path),
            "annotated_image": str(annotated_save_path),
            "prediction": detections,
        })
        pbar.update(1)

print(f"Inference completed: {len(prediction_records)} images")
print(f"Annotated images saved to: {annotated_dir.resolve()}")


In [ ]:

# CELL 25 — Save prediction summary
def make_json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if hasattr(obj, "tolist"):
        try:
            return obj.tolist()
        except Exception:
            pass
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)

predictions_json = INFERENCE_OUTPUT_DIR / "test_predictions.json"

with tqdm(total=1, desc="Saving predictions", unit="stage") as pbar:
    safe_predictions = make_json_safe(prediction_records)
    with predictions_json.open("w", encoding="utf-8") as f:
        json.dump(safe_predictions, f, indent=2)
    pbar.update(1)

print("Saved:", predictions_json)


In [ ]:

# CELL 26 — Visualization preview helper
def visualize_prediction(image_path, save_path=None):
    """
    Preview the annotated prediction image generated by Supervision.
    """
    img_path = Path(image_path)
    annotated_path = INFERENCE_OUTPUT_DIR / "annotated_images" / img_path.name
    target = annotated_path if annotated_path.exists() else img_path

    image = Image.open(target).convert("RGB")
    plt.figure(figsize=(12, 8))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"RF-DETR Prediction: {img_path.name}")
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()

print("Visualization helper ready.")
# Preview up to 3 test inferences if available
preview_count = min(3, len(test_images))
for p in test_images[:preview_count]:
    visualize_prediction(p)


In [ ]:

# CELL 27 — Final summary
summary = {
    "categories": CATEGORIES,
    "category_to_id": CATEGORY_TO_ID,
    "num_classes": NUM_CLASSES,
    "resolution": RESOLUTION,
    "device": DEVICE,
    "train_images": len(SPLITS["train"]),
    "val_images": len(SPLITS["val"]),
    "test_images": len(SPLITS["test"]),
    "train_annotations": dataset_summary["train"]["annotations"],
    "val_annotations": dataset_summary["val"]["annotations"],
    "test_annotations": dataset_summary["test"]["annotations"],
    "train_class_distribution": dataset_summary["train"]["class_distribution"],
    "val_class_distribution": dataset_summary["val"]["class_distribution"],
    "test_class_distribution": dataset_summary["test"]["class_distribution"],
    "output_dir": str(OUTPUT_DIR),
    "best_checkpoint": str(BEST_CHECKPOINT),
    "predictions_file": str(predictions_json),
    "annotated_images_dir": str(INFERENCE_OUTPUT_DIR / "annotated_images"),
}

print(json.dumps(summary, indent=2))
